# Semaine 7 — Jour 1 : n8n et frontières du workflow

Objectif : Identifier déclencheur, transformation et effet de bord dans un workflow n8n.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Pour une étude complète, prévoir 2 à 3 heures avec entretien, challenge et revue. Le premier lancement Docker peut demander un temps supplémentaire.

Prérequis : Python 3.12 recommandé, HTTP/JSON, notions d’agents et de persistance des semaines précédentes. Les commandes Python utilisent uniquement la bibliothèque standard. n8n exige Docker Compose v2 pour le lab d’intégration.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md) avant d’ouvrir [les corrigés](corriges/exercises_solution.md). Préparer [l’entretien](interview.md) et [le challenge](challenge.md).

Notebooks depuis la racine : `notebooks/week07/S7_J1_n8n_fondations.ipynb` et `notebooks/week07/teacher/S7_J1_n8n_fondations_teacher.ipynb`.

Critère de réussite : Décrire le contrat entre n8n et le backend, puis reproduire une réception durable.

Le lab ne requiert aucune clé LLM. Ses réponses sont simulées et ne mesurent pas la qualité d’un modèle réel.


# Objectifs mesurables

- Identifier déclencheur, transformation et effet de bord dans un workflow n8n.
- Démarrer un environnement local et distinguer URL de test et URL de production.
- Décrire le contrat entre n8n et le backend, puis reproduire une réception durable.

## Preuves attendues

Une sortie de lab reproductible, une explication de l’invariant étudié et une limite correctement identifiée. Le challenge approfondit le parcours essentiel sans être nécessaire pour finir la session de 60 minutes.


# Semaine 7 — Jour 1 : n8n et frontières du workflow


## 1. Le problème à résoudre
Une demande de support arrive depuis un formulaire. L’équipe souhaite l’enregistrer, préparer une réponse et demander une validation avant sa publication. Le workflow représente ce processus métier; les connecteurs exécutent les échanges entre systèmes. Ajouter un agent autonome pour piloter tout le parcours rendrait les règles de validation difficiles à contrôler. Nous réservons ici la génération du brouillon à un composant spécialisé et laissons le processus explicite.

n8n fournit une représentation visuelle des étapes et de leurs connexions. Un déclencheur reçoit une entrée; les nœuds manipulent des items et appellent des services. Un item porte généralement un objet `json`; un nœud peut produire plusieurs items. Ne supposez pas qu’une exécution contient toujours une seule demande. Pour ce lab, le contrat exige volontairement un objet événement par requête.

## 2. Les trois responsabilités
Le déclencheur détecte l’entrée du travail, la transformation prépare les données et l’effet modifie un système. Lire une facture n’a pas le même risque que publier une réponse. Dans notre exemple, recevoir l’événement écrit un job durable mais ne publie rien. Le service Python contrôle les transitions métier. n8n transporte les événements et déclenche périodiquement le traitement.

Ce choix permet de tester les invariants sans démarrer l’éditeur. Il est adapté à votre expérience backend : n8n joue le rôle d’un adaptateur d’intégration et le moteur celui d’un service applicatif. Il ne faut pas déplacer les règles de validation dans une succession d’expressions impossibles à tester. À l’inverse, coder tous les connecteurs en Python supprimerait une partie de l’intérêt de l’outil.

## 3. Premier parcours n8n
Importer le workflow `01_ingress.json`. Le nœud Webhook reçoit un POST et utilise une authentification par en-tête. Le nœud HTTP Request transmet `body` au backend. Respond to Webhook propage le résultat HTTP du backend. L’acceptation d’un événement nouveau produit 202; un doublon identique produit 200. Cette réponse confirme l’enregistrement, pas la fin du traitement.

L’URL de test est utilisée lorsque le nœud écoute une exécution manuelle. L’URL de production dépend d’un workflow publié ou activé selon l’interface de la version installée. Copier exactement l’URL affichée. Une erreur fréquente est d’envoyer à l’URL de test alors que personne n’écoute, ou d’oublier de rendre le workflow actif pour l’URL de production.

## 4. Environnement reproductible
La configuration Compose utilise des volumes séparés pour n8n et le backend. Le backend s’adresse à sa base SQLite; n8n conserve ses propres exécutions dans son volume. Ces bases ne sont pas interchangeables. Le nom DNS `backend` fonctionne entre conteneurs du projet Compose; `localhost` dans le conteneur n8n désigne n8n lui-même. Depuis votre terminal, utiliser `127.0.0.1:8080`.

Les tokens restent dans des credentials et dans le fichier `.env` local. L’export JSON ne contient aucune valeur secrète. Le lab utilise HTTP sur la boucle locale, avec cookies non sécurisés pour cette seule installation locale. Un environnement partagé demanderait TLS et une gestion d’identité différente.

## 5. Pourquoi commencer par un double déterministe ?
Le générateur fourni est une fonction à règles, explicitement marquée comme simulation. Il sépare le comportement du processus de la variabilité d’un modèle. Si un doublon publie deux fois une réponse, ce défaut existe indépendamment du fournisseur LLM. Aujourd’hui, observez l’acceptation; les jours suivants introduiront les transitions et les erreurs.

## À retenir
Une exécution n8n est un passage technique dans le graphe. Un job métier peut survivre à plusieurs exécutions. Les credentials, le contrat HTTP et l’état durable constituent la frontière entre ces deux niveaux.


## Diagramme du jour

```mermaid
flowchart TD
  A["Webhook n8n"] --> B["Adaptateur HTTP"]
  B --> C["Validation backend"]
  C --> D["Job durable"]
  C --> E["Réponse 202 ou 200"]
```

Si votre lecteur Jupyter ne rend pas Mermaid, ouvrir le fichier .mmd avec un lecteur compatible.

# Exemple exécutable autonome

Ce fragment illustre le concept du jour. Le moteur complet et ses scénarios sont dans examples/week07.



In [ ]:
payload = {"event_id": "evt-001", "type": "support.requested"}
accepted = set()
def accept(item):
    is_new = item["event_id"] not in accepted
    accepted.add(item["event_id"])
    return 202 if is_new else 200
assert accept(payload) == 202
assert accept(payload) == 200
print("Un seul identifiant accepté", accepted)


# Lab J1 — n8n et frontières du workflow

Depuis la racine du dépôt :

```bash
python3 book/week07/day01/labs/run.py
```

Résultat attendu : **accepted=true, status=received**.

Le script exécute le scénario du jour contre une base en mémoire et ferme la connexion. Il ne crée aucun message externe. Inspecter `examples/week07/demo.py` et la méthode correspondante dans `workflow.py` pour relier la sortie à la transition étudiée.

## Travail à réaliser

1. Exécuter le scénario et expliquer chaque champ retourné.
2. Réaliser l’exercice 2 du jour avec une entrée différente.
3. Justifier l’absence de publication non autorisée.

Pour le parcours n8n, suivre [le guide d’installation](../../../../docs/week07/QUICKSTART.md). J1 configure la réception; J2 ajoute le scheduler; J3 teste les doublons; J4 observe les reprises et erreurs; J5 étudie le port de génération; J6 importe le workflow de décision; J7 exécute la checklist d’intégration.

Le scénario J4 utilise une horloge virtuelle. Les tests de reprise sont rapides et n’attendent pas réellement les délais. Le projet HTTP utilise une base fichier pour démontrer la persistance après redémarrage.


# Exercices

Faire une première tentative sans consulter les corrigés.

## Exercice 1 — Repérer les responsabilités

Pour Webhook, HTTP Request, Respond to Webhook et Engine.ingest, préciser déclencheur, transport, réponse ou règle métier.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 2 — Comparer les accusés de réception

Expliquer pourquoi 202 ne garantit pas la publication et pourquoi un doublon reçoit 200.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 3 — Diagnostiquer le réseau

n8n appelle localhost:8080 dans Compose : expliquer l’erreur et donner la bonne URL.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.


# Préparation à l’entretien

Répondre en deux minutes par question, avec un exemple concret et une limite.

## Question 1

Quand choisir n8n plutôt qu’un service codé ?

## Question 2

Où stocker les invariants métier ?


# Challenge

Importer le workflow de réception, configurer ses credentials et envoyer deux fois le même événement. Conserver les deux réponses HTTP et une preuve qu’un seul job existe.

## Critères de revue

Reproductibilité, absence d’effet non autorisé, explication des limites et preuves du comportement attendu.


# Références — J1

Sources consultées le 12 octobre 2026. Les pages évoluent; les exports ciblent la version n8n indiquée dans Compose.

- [Documentation n8n](https://docs.n8n.io/) — vocabulaire et référence des nœuds.
- [Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) — réception et modes de réponse.
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) — appels au backend et credentials.
- [Gestion des erreurs](https://docs.n8n.io/flow-logic/error-handling/) — séparation du flux nominal et des erreurs techniques.
- [Wait](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait/) — variante d’attente, non implémentée ici.
- [sqlite3 Python](https://docs.python.org/3/library/sqlite3.html) — transactions et connexions.

Lire en priorité les éléments liés à l’objectif du jour; les autres liens permettent d’approfondir. Les choix du moteur pédagogique sont explicités dans le cours et ne constituent pas une spécification de n8n.


# Corrigé des exercices

## Exercice 1 — Repérer les responsabilités

Webhook déclenche; HTTP Request transporte; Respond to Webhook répond au demandeur; Engine.ingest valide et écrit dans une transaction. Le transport ne décide pas de la publication.

## Exercice 2 — Comparer les accusés de réception

202 signifie événement durablement accepté. 200 indique qu’un événement identique a déjà été accepté. Le client doit consulter le job pour connaître son état.

## Exercice 3 — Diagnostiquer le réseau

localhost est le conteneur n8n. Utiliser http://backend:8080/events depuis n8n et http://127.0.0.1:8080/events depuis le poste.


# Réponses d’entretien

## Question 1

Pour assembler rapidement des connecteurs et des règles de routage lisibles. Pour des invariants transactionnels et des traitements complexes, déléguer à un service testable. Le choix dépend de la charge, de la maintenance et des compétences.

## Question 2

Dans une frontière responsable de leur validation. Ici le backend vérifie le schéma et la déduplication; n8n ne possède pas une deuxième copie de ces règles.


# Solution du challenge

Exécuter le scénario curl décrit dans docs/week07/QUICKSTART.md. Attendre 202 puis 200. GET /metrics doit contenir un seul job pour cet event_id. Dans n8n, deux exécutions peuvent exister : ce nombre ne prouve pas deux jobs. Si Docker est indisponible, reproduire les deux POST directement sur le backend et marquer la vérification n8n comme non effectuée.


# Revue pédagogique

## Acquis à vérifier

- Identifier déclencheur, transformation et effet de bord dans un workflow n8n.
- Démarrer un environnement local et distinguer URL de test et URL de production.
- Décrire le contrat entre n8n et le backend, puis reproduire une réception durable.

## Erreurs fréquentes

Confondre réception et achèvement; assimiler un exemple simulé à une évaluation de modèle; déduire une garantie distribuée d’une transaction locale. Pour ce jour, vérifier en particulier : localhost est le conteneur n8n. Utiliser http://backend:8080/events depuis n8n et http://127.0.0.1:8080/events depuis le poste.

## Barème indicatif sur 10

- 3 points : explication des concepts avec le scénario du support.
- 3 points : reproduction du lab et analyse de sa sortie.
- 2 points : réponses aux exercices, sans confusion sur les garanties.
- 2 points : challenge argumenté et limite opérationnelle identifiée.

Le parcours essentiel peut être validé sans le challenge. Toute publication sans validation bloque la validation du projet intégré, quel que soit le score.


# Correction exécutable du lab

Dans Jupyter, démarrer le kernel depuis la racine du dépôt ou depuis son arbre. Ce fragment recherche la racine contenant le module S7. Il utilise le moteur complet, distinct du petit exemple autonome.



In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/week07/workflow.py").is_file()), None)
if root is None:
    raise RuntimeError("Ouvrir le notebook depuis le dépôt décompressé")
sys.path.insert(0, str(root))
from examples.week07.demo import scenario
result = scenario(1)
assert result == {'accepted': True, 'status': 'received'}
print(result)



Les assertions constituent la preuve minimale de ce scénario; les tests de `tests/week07` couvrent les branches complémentaires. Exécuter le scénario curl décrit dans docs/week07/QUICKSTART.md. Attendre 202 puis 200. GET /metrics doit contenir un seul job pour cet event_id. Dans n8n, deux exécutions peuvent exister : ce nombre ne prouve pas deux jobs. Si Docker est indisponible, reproduire les deux POST directement sur le backend et marquer la vérification n8n comme non effectuée.
